In [1]:
"""
m/z-to-m/z Spatial Pattern Matching Pipeline for Isotope Detection
===================================================================
Compares each m/z feature with all other m/z features within the same sample
to identify isotope patterns based on spatial similarity scores.

Based on Gene-to-m/z Spatial Pattern Matching Pipeline V1 (Analytic - Optimized)
Modified to:
- Compare m/z features within each sample (no cross-modal alignment)
- Remove 180-degree rotation (same coordinate space)
- Skip visualizations
- Output similarity scores to CSV
"""

import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.interpolate import griddata
from typing import Dict, Optional
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from joblib import Parallel, delayed
from itertools import combinations
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_PIXEL_SIZE = 60  # μm

# Mass differences for isotope and adduct detection
MASS_DIFFS = {
    'Isotope (M+1)': 1.0033,
    'Isotope (M+2)': 2.0067,
    'Adduct (NH4)': 17.0265,
    'Adduct (Na)': 21.982,
    'Adduct (K)': 37.9555
}

# Tolerance for mass difference matching (in Da)
MASS_DIFF_TOLERANCE = 0.01

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]


def compute_value_histogram(values: np.ndarray, n_bins: int = 50) -> np.ndarray:
    hist, _ = np.histogram(values, bins=n_bins, density=True)
    return hist / (hist.sum() + 1e-8)


def compute_spatial_histogram(coords: np.ndarray, values: np.ndarray, n_bins: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_bins).astype(int), 0, n_bins - 1)
    y_bins = np.clip((norm[:, 1] * n_bins).astype(int), 0, n_bins - 1)
    flat_idx = y_bins * n_bins + x_bins
    hist = np.bincount(flat_idx, weights=values, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    counts = np.bincount(flat_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    hist_min, hist_max = hist.min(), hist.max()
    if hist_max > hist_min:
        hist = (hist - hist_min) / (hist_max - hist_min)
    return hist


def compute_radial_profile(coords: np.ndarray, values: np.ndarray, n_rings: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    centroid = norm.mean(axis=0)
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() + 1e-8
    ring_idx = np.clip((distances / max_dist * n_rings).astype(int), 0, n_rings - 1)
    profile = np.bincount(ring_idx, weights=values, minlength=n_rings)
    counts = np.bincount(ring_idx, minlength=n_rings)
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile, dtype=float))
    prof_min, prof_max = profile.min(), profile.max()
    if prof_max > prof_min:
        profile = (profile - prof_min) / (prof_max - prof_min)
    return profile


def compute_quadrant_stats(coords: np.ndarray, values: np.ndarray, n_div: int = 3) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_div).astype(int), 0, n_div - 1)
    y_bins = np.clip((norm[:, 1] * n_div).astype(int), 0, n_div - 1)
    flat_idx = y_bins * n_div + x_bins
    stats = np.zeros(n_div * n_div * 2)
    for q in range(n_div * n_div):
        mask = flat_idx == q
        if mask.sum() > 0:
            q_vals = values[mask]
            stats[q * 2] = np.mean(q_vals)
            stats[q * 2 + 1] = np.std(q_vals)
    stats_min, stats_max = stats.min(), stats.max()
    if stats_max > stats_min:
        stats = (stats - stats_min) / (stats_max - stats_min)
    return stats


def compute_morans_i_vectorized(coords: np.ndarray, values: np.ndarray, indices: np.ndarray) -> float:
    n = len(values)
    mean_val = values.mean()
    deviations = values - mean_val
    denom = np.sum(deviations ** 2)
    if denom == 0:
        return 0.0
    neighbor_deviations = deviations[indices[:, 1:]]
    numer = np.sum(deviations[:, np.newaxis] * neighbor_deviations)
    w_sum = indices.shape[0] * (indices.shape[1] - 1)
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0.0


@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    value_histogram: np.ndarray = None
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    quadrant_stats: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None


def compute_coordinate_based_similarity(sig1: SpatialSignature, sig2: SpatialSignature, grid_res: int = 50) -> dict:
    """Compute coordinate-based similarity between two signatures in the same coordinate space."""
    coords = sig1.coordinates  # Same coordinates for both (same sample)
    
    # Direct value correlation (no grid interpolation needed - same coordinates)
    mask = (sig1.raw_values > 0) | (sig2.raw_values > 0)
    value_corr = 0
    if mask.sum() > 10:
        r, _ = pearsonr(sig1.raw_values, sig2.raw_values)
        value_corr = r if not np.isnan(r) else 0
    
    # Importance correlation
    imp_corr = 0
    if len(sig1.node_importance) > 10:
        r, _ = pearsonr(sig1.node_importance, sig2.node_importance)
        imp_corr = r if not np.isnan(r) else 0
    
    # Importance IoU
    imp1 = sig1.node_importance / (sig1.node_importance.max() + 1e-8)
    imp2 = sig2.node_importance / (sig2.node_importance.max() + 1e-8)
    importance_iou = np.minimum(imp1, imp2).sum() / (np.maximum(imp1, imp2).sum() + 1e-8)
    
    return {'value_correlation': value_corr, 'importance_iou': importance_iou, 'importance_correlation': imp_corr}


def compute_descriptor_similarity(sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
    def safe_pearsonr(a, b):
        if a.std() > 0 and b.std() > 0:
            r, _ = pearsonr(a, b)
            return r if not np.isnan(r) else 0
        return 0
    
    val_hist_corr = safe_pearsonr(sig1.value_histogram, sig2.value_histogram)
    spatial_hist_corr = safe_pearsonr(sig1.spatial_histogram.flatten(), sig2.spatial_histogram.flatten())
    radial_corr = safe_pearsonr(sig1.radial_profile, sig2.radial_profile)
    quad_corr = safe_pearsonr(sig1.quadrant_stats, sig2.quadrant_stats)
    morans_sim = 1 / (1 + abs(sig1.morans_i - sig2.morans_i))
    
    return {'value_hist_corr': val_hist_corr, 'spatial_hist_corr': spatial_hist_corr,
            'radial_corr': radial_corr, 'quadrant_corr': quad_corr, 'morans_similarity': morans_sim}


def compute_combined_score(coord_sim: dict, desc_sim: dict) -> float:
    coord_score = (0.1590 * max(coord_sim['value_correlation'], 0) +
                   0.1225 * coord_sim['importance_iou'] +
                   0.1590 * max(coord_sim['importance_correlation'], 0))
    desc_score = (0.1504 * max(desc_sim['spatial_hist_corr'], 0) +
                  0.1590 * max(desc_sim['radial_corr'], 0) +
                  0.0807 * max(desc_sim['quadrant_corr'], 0) +
                  0.0280 * desc_sim['morans_similarity'] +
                  0.1416 * max(desc_sim['value_hist_corr'], 0))
    return coord_score + desc_score


def classify_mass_difference(mz_diff: float, tolerance: float = MASS_DIFF_TOLERANCE) -> str:
    """Classify the mass difference into isotope/adduct type."""
    for label, expected_diff in MASS_DIFFS.items():
        if abs(mz_diff - expected_diff) <= tolerance:
            return label
    return 'None'


class MzIsotopeMatcher:
    def __init__(self, output_dir: str = './mz_isotope_matching_results', n_jobs: int = -1):
        self.output_dir = output_dir
        self.n_jobs = n_jobs
        os.makedirs(output_dir, exist_ok=True)
        self.msi_data = {}
        self._nn_cache = {}

    def load_all_data(self):
        print(f"Loading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
            else:
                print(f"  {sample_id}: NOT FOUND at {path}")

    def _get_nn_indices(self, coords: np.ndarray, k: int, cache_key: str) -> np.ndarray:
        full_key = f"{cache_key}_{k}"
        if full_key not in self._nn_cache:
            coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
            norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
            k_actual = min(k + 1, len(coords))
            nn = NearestNeighbors(n_neighbors=k_actual)
            nn.fit(norm)
            _, indices = nn.kneighbors(norm)
            self._nn_cache[full_key] = indices
        return self._nn_cache[full_key]

    def compute_bio_importance(self, coords: np.ndarray, values: np.ndarray, k: int, nn_indices: np.ndarray) -> np.ndarray:
        neighbor_vals = values[nn_indices[:, 1:]]
        local_var = np.var(neighbor_vals, axis=1)
        lv_min, lv_max = local_var.min(), local_var.max()
        if lv_max > lv_min:
            local_var = (local_var - lv_min) / (lv_max - lv_min)
        else:
            local_var = np.zeros_like(local_var)
        v_min, v_max = values.min(), values.max()
        if v_max > v_min:
            val_norm = (values - v_min) / (v_max - v_min)
        else:
            val_norm = np.zeros_like(values)
        return 0.5 * local_var + 0.5 * val_norm

    def extract_signature(self, coords: np.ndarray, values: np.ndarray, sample_id: str,
                          feature_name: str, n_neighbors: int, nn_indices: np.ndarray) -> SpatialSignature:
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors, nn_indices)
        return SpatialSignature(
            sample_id=sample_id, feature_name=feature_name, feature_type='mz',
            node_importance=bio_imp, value_histogram=compute_value_histogram(values),
            spatial_histogram=compute_spatial_histogram(coords, values),
            radial_profile=compute_radial_profile(coords, values),
            quadrant_stats=compute_quadrant_stats(coords, values),
            morans_i=compute_morans_i_vectorized(coords, values, nn_indices),
            coordinates=coords, raw_values=values)

    def compute_pair_similarity(self, sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
        """Compute all similarity metrics between two m/z signatures."""
        coord_sim = compute_coordinate_based_similarity(sig1, sig2)
        desc_sim = compute_descriptor_similarity(sig1, sig2)
        combined = compute_combined_score(coord_sim, desc_sim)
        
        # Calculate m/z difference for isotope detection
        try:
            mz1 = float(sig1.feature_name)
            mz2 = float(sig2.feature_name)
            mz_diff = abs(mz2 - mz1)
            # Classify the mass difference
            mass_diff_type = classify_mass_difference(mz_diff)
        except ValueError:
            mz_diff = np.nan
            mass_diff_type = 'None'
        
        return {
            'mz_1': sig1.feature_name,
            'mz_2': sig2.feature_name,
            'mz_difference': mz_diff,
            'mass_diff_type': mass_diff_type,
            'sample_id': sig1.sample_id,
            **coord_sim,
            **desc_sim,
            'combined_score': combined,
            'mz_1_morans_i': sig1.morans_i,
            'mz_2_morans_i': sig2.morans_i
        }

    def run_analysis(self, top_k_per_mz: int = 20):
        print("\n" + "=" * 70)
        print("m/z-to-m/z MATCHING FOR ISOTOPE DETECTION")
        print(f"MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print("Strategy: Analytic Spatial Descriptors + Heuristic Importance")
        print("=" * 70)

        all_results = []
        all_top_matches = []

        for sample_id in tqdm(MSI_SAMPLE_IDS, desc="Samples", unit="sample"):
            if sample_id not in self.msi_data:
                print(f"\n{sample_id}: NOT LOADED, skipping")
                continue

            print(f"\n{'=' * 50}")
            print(f"Sample: {sample_id}")
            print(f"{'=' * 50}")

            msi_adata = self.msi_data[sample_id]
            msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
            msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
            mz_names = list(msi_adata.var_names)
            n_mz = len(mz_names)

            print(f"  {n_mz} m/z features, {len(msi_coords)} pixels")

            # Pre-compute nearest neighbors
            nn_indices = self._get_nn_indices(msi_coords, 8, f"msi_{sample_id}")

            # Extract all signatures
            print(f"  Extracting signatures...")
            def extract_single_mz(i):
                return self.extract_signature(msi_coords, msi_data[:, i], sample_id, mz_names[i], 8, nn_indices)

            signatures = Parallel(n_jobs=self.n_jobs, prefer='threads')(
                delayed(extract_single_mz)(i) for i in tqdm(range(n_mz), desc="  Signatures", unit="mz"))
            
            sig_dict = {mz_names[i]: signatures[i] for i in range(n_mz)}
            print(f"  {n_mz} signatures extracted")

            # Compare all pairs
            print(f"  Computing pairwise similarities...")
            n_pairs = n_mz * (n_mz - 1) // 2
            print(f"  {n_pairs} pairs to compare")

            def compute_pair(i, j):
                return self.compute_pair_similarity(signatures[i], signatures[j])

            # Generate all pairs
            pair_indices = list(combinations(range(n_mz), 2))
            
            # Parallel computation with progress bar
            pair_results = Parallel(n_jobs=self.n_jobs, prefer='threads')(
                delayed(compute_pair)(i, j) for i, j in tqdm(pair_indices, desc="  Pairs", unit="pair"))

            # Convert to DataFrame
            sample_results = pd.DataFrame(pair_results)
            all_results.append(sample_results)

            # Find top matches for each m/z
            print(f"  Finding top {top_k_per_mz} matches per m/z...")
            for mz_name in tqdm(mz_names, desc="  Top matches", unit="mz"):
                # Get all pairs involving this m/z
                mz_pairs = sample_results[
                    (sample_results['mz_1'] == mz_name) | (sample_results['mz_2'] == mz_name)
                ].copy()
                
                # Sort by combined score
                mz_pairs = mz_pairs.sort_values('combined_score', ascending=False)
                
                # Take top k
                top_matches = mz_pairs.head(top_k_per_mz).copy()
                top_matches['query_mz'] = mz_name
                top_matches['rank'] = range(1, len(top_matches) + 1)
                all_top_matches.append(top_matches)

            print(f"  Sample complete: {len(sample_results)} pairs scored")

        # Save all results
        if all_results:
            print("\n" + "=" * 70)
            print("SAVING RESULTS")
            print("=" * 70)

            # Full pairwise results
            full_results = pd.concat(all_results, ignore_index=True)
            full_results = full_results.sort_values(['sample_id', 'combined_score'], ascending=[True, False])
            full_path = os.path.join(self.output_dir, 'mz_to_mz_all_pairs.csv')
            full_results.to_csv(full_path, index=False)
            print(f"  All pairs: {full_path} ({len(full_results)} rows)")

            # Top matches per m/z
            if all_top_matches:
                top_df = pd.concat(all_top_matches, ignore_index=True)
                priority_cols = ['sample_id', 'query_mz', 'rank', 'mz_1', 'mz_2', 'mz_difference', 'mass_diff_type', 'combined_score']
                other_cols = [c for c in top_df.columns if c not in priority_cols]
                top_df = top_df[priority_cols + other_cols]
                top_df = top_df.sort_values(['sample_id', 'query_mz', 'rank'])
                top_path = os.path.join(self.output_dir, f'mz_to_mz_top{top_k_per_mz}_matches.csv')
                top_df.to_csv(top_path, index=False)
                print(f"  Top matches: {top_path} ({len(top_df)} rows)")

            # Summary statistics
            summary = []
            for sample_id in MSI_SAMPLE_IDS:
                if sample_id in self.msi_data:
                    sample_data = full_results[full_results['sample_id'] == sample_id]
                    summary.append({
                        'sample_id': sample_id,
                        'n_mz_features': len(self.msi_data[sample_id].var_names),
                        'n_pairs': len(sample_data),
                        'mean_combined_score': sample_data['combined_score'].mean(),
                        'max_combined_score': sample_data['combined_score'].max(),
                        'min_combined_score': sample_data['combined_score'].min(),
                        'std_combined_score': sample_data['combined_score'].std()
                    })
            
            summary_df = pd.DataFrame(summary)
            summary_path = os.path.join(self.output_dir, 'mz_matching_summary.csv')
            summary_df.to_csv(summary_path, index=False)
            print(f"  Summary: {summary_path}")

            # Potential isotope/adduct pairs based on mass differences
            isotope_adduct_candidates = full_results[
                (full_results['mass_diff_type'] != 'None') &
                (full_results['combined_score'] >= 0.5)
            ].copy()
            isotope_adduct_candidates = isotope_adduct_candidates.sort_values(
                ['mass_diff_type', 'combined_score'], ascending=[True, False])
            isotope_path = os.path.join(self.output_dir, 'potential_isotope_adduct_pairs.csv')
            isotope_adduct_candidates.to_csv(isotope_path, index=False)
            print(f"  Potential isotopes/adducts: {isotope_path} ({len(isotope_adduct_candidates)} pairs)")

            # Print summary by type
            if len(isotope_adduct_candidates) > 0:
                print("\n  Detected pairs by type:")
                for diff_type in MASS_DIFFS.keys():
                    type_count = len(isotope_adduct_candidates[
                        isotope_adduct_candidates['mass_diff_type'] == diff_type])
                    if type_count > 0:
                        print(f"    {diff_type}: {type_count} pairs")

            return full_results
        
        return None


def main():
    print("=" * 70)
    print("m/z-to-m/z Isotope Pattern Matching")
    print(f"MSI: {MSI_PIXEL_SIZE}μm")
    print("=" * 70)
    
    matcher = MzIsotopeMatcher(output_dir='./synced_mz_isotope_matching_results', n_jobs=-1)
    matcher.load_all_data()
    results = matcher.run_analysis(top_k_per_mz=20)
    
    print("\n" + "=" * 70)
    print("COMPLETE!")
    print("=" * 70)
    
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()

m/z-to-m/z Isotope Pattern Matching
MSI: 60μm
Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

m/z-to-m/z MATCHING FOR ISOTOPE DETECTION
MSI: 60μm (Cartesian)
Strategy: Analytic Spatial Descriptors + Heuristic Importance


Samples:   0%|          | 0/16 [00:00<?, ?sample/s]


Sample: YC_1
  528 m/z features, 6688 pixels
  Extracting signatures...


  Signatures: 100%|██████████| 528/528 [00:01<00:00, 433.76mz/s]


  528 signatures extracted
  Computing pairwise similarities...
  139128 pairs to compare


Samples:   0%|          | 0/16 [03:00<?, ?sample/s]


KeyboardInterrupt: 